In [ ]:
# 2. FEATURE ENGINEERING — OLS -> filtre -> Lasso, par typologie
from pathlib import Path
import pandas as pd
import statsmodels.api as sm
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LassoCV
from sklearn.compose import ColumnTransformer
import json


ROOT = Path("..").resolve()
DATA = ROOT / "data"
DATA.mkdir(exist_ok=True, parents=True)

SRC = DATA / "data_wip_v5.csv"
DST = DATA / "df_dummies_2019.csv"  # existe peut-être déjà
assert SRC.exists(), f"Introuvable : {SRC}"

df = pd.read_csv(SRC, sep=";", encoding="utf-8")
print(df.shape)
df.head(2)

TARGETS = [
    "Déchets_verts",
    "Matériaux_recyclables",
    "Encombrants",
    "Total_autres_dechets",
    "Déblais_gravats",
]
for t in TARGETS:
    assert t in df.columns, f"Colonne cible manquante : {t}"

In [ ]:
# Colonnes explicatives de départ = tout sauf les 5 cibles et les identifiants de structure
id_cols = [
    c for c in ["année", "annee", "Région", "region", "Code_Dpt"] if c in df.columns
]
X_base = df.drop(columns=TARGETS + id_cols)
cat_cols = X_base.select_dtypes(include="object").columns.tolist()
num_cols = X_base.select_dtypes(exclude="object").columns.tolist()
print("Catégorielles:", len(cat_cols), "| Numériques:", len(num_cols))

# Pipeline d'encodage (OneHot sur catégorielles)
preproc = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), cat_cols)
    ],
    remainder="passthrough",  # garde les numériques telles quelles
)


def select_by_ols_then_lasso(X_df: pd.DataFrame, y: pd.Series, p_th=0.05):
    # 1) Fit preproc pour obtenir X encodé (dataframe)
    X_enc = preproc.fit_transform(X_df)
    # Récupérer noms de colonnes encodées
    ohe = preproc.named_transformers_["cat"]
    ohe_names = ohe.get_feature_names_out(cat_cols).tolist() if cat_cols else []
    feat_names = ohe_names + num_cols
    X_enc_df = pd.DataFrame(X_enc, columns=feat_names, index=X_df.index)

    # 2) OLS -> filtrage p-values
    Xc = sm.add_constant(X_enc_df, has_constant="add")
    ols = sm.OLS(y, Xc, missing="drop").fit()
    pvals = ols.pvalues.drop(labels=["const"], errors="ignore")
    sig_feats = pvals[pvals < p_th].index.tolist()
    if not sig_feats:
        # si rien < 0.05, on prend le top 50 plus significatifs (ou tout si <50)
        sig_feats = pvals.sort_values().head(min(50, len(pvals))).index.tolist()

    # 3) LassoCV sur sous-ensemble significatif
    lasso = LassoCV(cv=5, random_state=42, max_iter=20000)
    lasso.fit(X_enc_df[sig_feats], y)
    kept = pd.Series(lasso.coef_, index=sig_feats)
    kept_feats = kept[kept != 0].index.tolist()
    return kept_feats, feat_names

In [ ]:
# Boucle par cible -> features retenues
features_by_target = {}
for target in TARGETS:
    print("=== Cible:", target)
    y = pd.to_numeric(df[target], errors="coerce")
    kept_feats, all_names = select_by_ols_then_lasso(X_base, y, p_th=0.05)
    print(" -> features retenues:", len(kept_feats))
    features_by_target[target] = kept_feats

# Sauvegarde pour traçabilité

(DATA / "features_by_target.json").write_text(
    json.dumps(features_by_target, ensure_ascii=False, indent=2), encoding="utf-8"
)
features_by_target

In [ ]:
# (Option) Régénérer df_dummies_2019.csv en union des features retenues
regen = False  # passe à True si tu veux reconstruire le CSV final
if regen:
    final_feats = sorted(set().union(*features_by_target.values()))
    # Re-générer X_enc complet pour ces colonnes
    X_enc = preproc.fit_transform(X_base)
    ohe = preproc.named_transformers_["cat"]
    ohe_names = ohe.get_feature_names_out(cat_cols).tolist() if cat_cols else []
    feat_names = ohe_names + num_cols
    X_enc_df = pd.DataFrame(X_enc, columns=feat_names, index=X_base.index)
    df_dum = pd.concat(
        [
            df[TARGETS].reset_index(drop=True),
            X_enc_df[final_feats].reset_index(drop=True),
        ],
        axis=1,
    )
    df_dum.to_csv(DST, index=False, encoding="utf-8")
    print("Saved:", DST)
else:
    print("Pas de régénération — on garde le df_dummies_2019.csv existant.")